# Week 2 Option A — portable notebook
Open this notebook from the downloaded project folder. Outputs have been removed for sharing.


## Cell 1 — Install tools


In [ ]:
%pip install -q gradio==6.30.0 scikit-learn pandas requests
print("Installation finished.")

## Cell 2 — Check local project files


In [ ]:
from pathlib import Path
import os
app_folder = Path.cwd()
assert (app_folder / "app/config.py").exists(), "Open this notebook from the project folder."
assert (app_folder / "my_labels.csv").exists(), "The 30-label answer sheet is required."
print("Everything is in place.")


## Cell 3 — Optional private API key
Enter a key only in the hidden prompt. Never put it in notebook source or saved outputs.


In [ ]:
import os
from getpass import getpass
if not os.environ.get("GROQ_API_KEY", "").strip():
    entered_key = getpass("API key (hidden; leave blank for demo): ").strip()
    if entered_key:
        os.environ["GROQ_API_KEY"] = entered_key
    del entered_key
print("Key loaded." if os.environ.get("GROQ_API_KEY") else "Demo mode selected.")


## Cell 4 — Experiment settings


In [ ]:
%%writefile app/config.py

SYSTEM_PROMPT = """
Sort each news item into exactly one of these four categories.

World: politics, government, international affairs, conflict and disasters.
Sports: matches, results, athletes, teams, transfers and tournaments.
Business: companies, markets, earnings, the economy, trade and commercial deals.
Sci/Tech: science, research, technology, software, space and health research.
If two categories fit, choose the one the headline is mainly about instead of relying on a single keyword.
Choose Business for stories mainly about company earnings, share prices or commercial deals, even when the company operates in technology or sport
If two categories seem equally plausible, choose the best-fitting one and report confidence of 0.4 or lower.
"""


ROUTING_RULES = [
    ("why did you",        "explain"),
    ("explain",            "explain"),
    ("how confident",      "explain"),
    ("ignore your",        "refuse"),
    ("your instructions",  "refuse"),
    ("weather",            "refuse"),
]

BANNED_PHRASES = [
    "as a human",
    "i am a person",
]

ALLOWED_CATEGORIES = ["World", "Sports", "Business", "Sci/Tech"]

CONFIDENCE_FLOOR = 0.4

KNOWLEDGE_FILES = [
    "categories.md",
    "escalation.md"
]

KNOWLEDGE_PASSAGES = 2
KNOWLEDGE_THRESHOLD = 0.10

MODEL = "openai/gpt-oss-120b"
TEMPERATURE = 0

## Cell 5 — Start the app
The sharing option creates a temporary public app link while the notebook runs.


In [ ]:
import gradio as gr
import sys

gr.close_all()

for name in list(sys.modules):
    if name == "app" or name.startswith("app."):
        del sys.modules[name]

from app.ui import build
build().launch(share=True)

## Cell 6 — Score against the 30 personal Week 1 labels


In [ ]:
import pandas as pd
from app import engine
df = pd.read_csv("my_labels.csv")
golden = [{"text": r.text, "label": r.my_label} for r in df.itertuples()]
s = engine.score(golden)
print(f"Correct: {s['correct']} out of {s['total']}")
print(f"Accuracy: {s['accuracy']*100:.1f}%")
print(f"Kappa: {s['kappa']}")
